In [3]:
import numpy as np 
import pandas as pd  

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, brier_score_loss


from get_train_test import TrainTestBuilder


In [ ]:

fp = r'C:\Users\jcmar\my_files\SportsBetting\data\training_data\entire_odds_stats_2026-03-07.csv'
df_model = pd.read_csv(fp)

df_model['math_red'] = df_model['math_red'].astype('category')
df_model['math_blue'] = df_model['math_blue'].astype('category')
df_model['elo_pred'] = df_model['elo_pred'].astype('category')

non_feats = [
    'date','event_date','event_location','fighter_blue','fighter_red',
    'method','og_blue_name','og_red_fighter', 'red_fighter_stats', 'blue_fighter_stats',
    'pimp_close1_blue','pimp_close1_red','pimp_close2_blue','pimp_close2_red',
    'juice_close1_blue','juice_close1_red','juice_close2_blue','juice_close2_red',
    'winner_name',
    
    'red_fighter_odds','blue_fighter_odds',
    'dec_close1_blue','dec_close1_red','dec_close2_blue','dec_close2_red',
    'dec_fair_close1_blue','dec_fair_close1_red','dec_fair_close2_blue','dec_fair_close2_red',
    'red_ud_to_fav_close1','red_ud_to_fav_close2','blue_ud_to_fav_close1','blue_ud_to_fav_close2',
    'red_stayed_fav_close1','red_stayed_fav_close2','blue_stayed_fav_close1','blue_stayed_fav_close2',
    'red_fav_to_ud_close1','red_fav_to_ud_close2','blue_fav_to_ud_close1','blue_fav_to_ud_close2',
    'red_stayed_dog_close1','red_stayed_dog_close2','blue_stayed_dog_close1','blue_stayed_dog_close2',
    'performance_bonus_winner', 'fight_otn_bonus'
] 


selected_feats = [
                  'proba_fair_close2_diff', 'proba_fair_open_diff', 'reach_diff', 
                  
                  'sub_att_pm_red', 'sub_att_pm_blue',
                  'ratio_control_diff',

                  'td_landed_pm_diff',  
                  'ratio_td_diff', 
                  'adjusted_td_red', 'adjusted_td_blue',

                  'sig_str_absorbed_total_diff', 
                  'sig_str_accuracy_pct_diff',
                  'sig_str_defense_pct_diff',
                  'adjusted_sig_str_blue', 'adjusted_sig_str_red', 
                  
                  'win_pct_red', 'win_pct_blue',
                  'win_streak_diff', 'lose_streak_diff',
                  'elo_red', 'elo_blue', 'elo_pred', 'age_red', 'age_blue',
                  ]

outlier_dict = None
y = 'winner'
builder = TrainTestBuilder(df=df_model, target_col=y, non_features=non_feats, train_size=0.85, random_state=42)
builder.filter_by_date(year=2010, month=2, day=26, date_col='event_date')
X_train, X_test, y_train, y_test, df_train, df_test, scaler_open = builder.prepare_train_test(selected_feats, scale=False, outlier_dict=outlier_dict)


Filtered: kept 6650 rows from 2010-02-26 onward.
PREPARE SHAPE: (4256, 297)
MODEL SHAPE: (4177, 297)
Categorical columns: ['elo_pred']
Numerical columns: ['proba_fair_close2_diff', 'proba_fair_open_diff', 'reach_diff', 'sub_att_pm_red', 'sub_att_pm_blue', 'ratio_control_diff', 'td_landed_pm_diff', 'ratio_td_diff', 'adjusted_td_red', 'adjusted_td_blue', 'sig_str_absorbed_total_diff', 'sig_str_accuracy_pct_diff', 'sig_str_defense_pct_diff', 'adjusted_sig_str_blue', 'adjusted_sig_str_red', 'win_pct_red', 'win_pct_blue', 'win_streak_diff', 'lose_streak_diff', 'elo_red', 'elo_blue', 'age_red', 'age_blue']


In [35]:
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
X_poly = poly.fit_transform(X_train)
print(X_poly.shape)


(3550, 276)


In [4]:
pipe = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scale_poly", StandardScaler()),
    ("select", SelectKBest(score_func=f_classif)),
    ("model", LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=5000
    ))
])


param_grid = {
    # overall polynomial degree
    "poly__degree": [1, 2, 3],

    # whether to include interactions
    "poly__interaction_only": [True, False],

    # how many transformed features to keep
    "select__k": [20, 25, 15],
    "model__C": [3.5]
}


grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=10,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

poly = best_model.named_steps["poly"]
selector = best_model.named_steps["select"]
                                  
feature_names = poly.get_feature_names_out(
    X_train.columns
)

mask = selector.get_support()
selected_features = feature_names[mask]

print(selected_features)
print(len(selected_features))
print(feature_names)

['proba_fair_close2_diff' 'proba_fair_open_diff' 'reach_diff'
 'sub_att_pm_red' 'ratio_control_diff' 'td_landed_pm_diff' 'ratio_td_diff'
 'adjusted_td_red' 'adjusted_td_blue' 'sig_str_absorbed_total_diff'
 'sig_str_accuracy_pct_diff' 'adjusted_sig_str_blue' 'win_pct_red'
 'win_pct_blue' 'win_streak_diff' 'lose_streak_diff' 'elo_red' 'age_red'
 'age_blue' 'elo_pred']
20
['proba_fair_close2_diff' 'proba_fair_open_diff' 'reach_diff'
 'sub_att_pm_red' 'sub_att_pm_blue' 'ratio_control_diff'
 'td_landed_pm_diff' 'ratio_td_diff' 'adjusted_td_red' 'adjusted_td_blue'
 'sig_str_absorbed_total_diff' 'sig_str_accuracy_pct_diff'
 'sig_str_defense_pct_diff' 'adjusted_sig_str_blue' 'adjusted_sig_str_red'
 'win_pct_red' 'win_pct_blue' 'win_streak_diff' 'lose_streak_diff'
 'elo_red' 'elo_blue' 'age_red' 'age_blue' 'elo_pred']


In [5]:
print(grid.best_params_)
print(grid.best_score_)

{'model__C': 3.5, 'poly__degree': 1, 'poly__interaction_only': True, 'select__k': 20}
0.7017212158503288


In [6]:

y_pred = best_model.predict(X_test)

acc = accuracy_score(
    y_test,
    y_pred
)

print(acc)

0.7049441786283892


In [8]:
pipe = Pipeline([
    ("spline", SplineTransformer(include_bias=False)),
    ("scale_spline", StandardScaler()),
    ("select", SelectKBest(score_func=f_classif)),
    (
        "model",
        LogisticRegression(
            penalty="l1",          # lasso penalty
            solver="saga",         # required for l1
            max_iter=5000
        )
    )
])


param_grid = {
    # spline degree
    "spline__degree": [4,2,3],

    # number of knots
    "spline__n_knots": [2, 3, 5],

    # knot placement
    "spline__knots": ["uniform", "quantile"],

    # number of transformed spline features to keep
    "select__k": [25, 20, 15],
    "model__C": [3.5]
}


grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=10,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)

# best fitted pipeline
best_model = grid.best_estimator_

# pipeline steps
spline = best_model.named_steps["spline"]
selector = best_model.named_steps["select"]

# get spline-transformed feature names
feature_names = spline.get_feature_names_out(
    X_train.columns
)

# boolean mask from SelectKBest
mask = selector.get_support()

# selected spline features
selected_features = feature_names[mask]

print(selected_features)
print(len(selected_features))



['proba_fair_close2_diff_sp_0' 'proba_fair_close2_diff_sp_1'
 'proba_fair_open_diff_sp_0' 'reach_diff_sp_0' 'ratio_control_diff_sp_0'
 'ratio_control_diff_sp_1' 'td_landed_pm_diff_sp_0'
 'td_landed_pm_diff_sp_1' 'ratio_td_diff_sp_0' 'adjusted_td_red_sp_0'
 'adjusted_td_red_sp_1' 'sig_str_absorbed_total_diff_sp_0'
 'sig_str_accuracy_pct_diff_sp_0' 'adjusted_sig_str_blue_sp_1'
 'win_pct_red_sp_0' 'win_pct_red_sp_1' 'win_pct_blue_sp_1'
 'win_streak_diff_sp_0' 'win_streak_diff_sp_1' 'lose_streak_diff_sp_0'
 'elo_red_sp_0' 'age_red_sp_0' 'age_blue_sp_0' 'age_blue_sp_1'
 'elo_pred_sp_0']
25


c:\Users\jcmar\my_files\SportsBetting\venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [47] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\jcmar\my_files\SportsBetting\venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [9]:
print(grid.best_params_)
print(grid.best_score_)

{'model__C': 3.5, 'select__k': 25, 'spline__degree': 2, 'spline__knots': 'quantile', 'spline__n_knots': 2}
0.702808530197222


In [10]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

acc = accuracy_score(
    y_test,
    y_pred
)

print(acc)
print(brier_score_loss(y_test, y_proba[:,1]))



0.7033492822966507
0.1985011636047109


In [125]:
lr = LogisticRegression(max_iter=1000, penalty='l1', solver="saga", C=5.5)
lr.fit(X_train, y_train)
y_hat = lr.predict(X_test)
y_proba = lr.predict_proba(X_test)

y_hat_train = lr.predict(X_train)

print(accuracy_score(y_test,y_hat))
print(accuracy_score(y_train, y_hat_train))
print(brier_score_loss(y_test, y_proba[:,1]))

0.7001594896331739
0.6552112676056338
0.19460288721118682
